# FraudGuard AI — Google Colab GPU Model Training
### Multilingual Financial Fraud & Scam-Pattern Advisory System
**Capstone Project — B.Tech CSE (7th Semester)**

This notebook allows you to train deep neural Transformer classifiers (`sentence-transformers` or `IndicBERT/MuRIL`) using a free Google Colab GPU (T4).

**Workflow:**
1. Connect to GPU (`Runtime` -> `Change runtime type` -> `T4 GPU`).
2. Upload your `data/splits/train.csv`, `val.csv`, and `test.csv`.
3. Fine-tune a Transformer model.
4. Evaluate and generate classification reports.
5. Download the model artifact to place into your local `models/advanced_classifier/` folder.

In [ ]:
# 1. Install required dependencies
!pip install -q transformers datasets sentence-transformers accelerate evaluate scikit-learn pandas matplotlib seaborn

In [ ]:
# 2. Check GPU availability
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# 3. Load dataset splits
import pandas as pd

# If running in Colab, you can upload train.csv, val.csv, test.csv from your data/splits/ directory
from google.colab import files
print("Please upload train.csv, val.csv, test.csv:")
uploaded = files.upload()

train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('val.csv')
test_df = pd.read_csv('test.csv')

print(f"Loaded {len(train_df)} train, {len(val_df)} val, {len(test_df)} test samples.")

In [ ]:
# 4. Prepare label mappings
labels = sorted(train_df['category'].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

train_df['label'] = train_df['category'].map(label2id)
val_df['label'] = val_df['category'].map(label2id)
test_df['label'] = test_df['category'].map(label2id)

print("Number of classes:", len(labels))
print("Classes:", labels)

In [ ]:
# 5. Tokenize using Transformer (e.g. distilbert or all-MiniLM-L6-v2)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize_fn, batched=True)
val_dataset = Dataset.from_pandas(val_df[['text', 'label']]).map(tokenize_fn, batched=True)
test_dataset = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
# 6. Configure Trainer and Fine-Tune
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "macro_f1": macro_f1}

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=10,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting GPU fine-tuning...")
trainer.train()

In [ ]:
# 7. Final Held-Out Test Set Evaluation
test_preds = trainer.predict(test_dataset)
y_preds = np.argmax(test_preds.predictions, axis=1)
y_true = test_df['label'].values

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("=== Final Held-Out Test Report ===")
print(classification_report(y_true, y_preds, target_names=labels, zero_division=0))

cm = confusion_matrix(y_true, y_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Colab Fine-Tuned Transformer: Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# 8. Save and Download Model Artifact
save_path = "./fraudguard_transformer_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Zip for download
!zip -r fraudguard_transformer_model.zip ./fraudguard_transformer_model

from google.colab import files
files.download('fraudguard_transformer_model.zip')
print("Downloaded! Extract contents into models/advanced_classifier/ in your repository.")